In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [19]:
df=pd.read_csv("C:/Users/Durga/OneDrive/Desktop/DataScience/ExpenseAnalyser/OpTransactionHistory26-04-2026.csv",encoding="ISO-8859-1")


In [20]:
df

,ï»¿S No.,Value Date,Transaction Date,Cheque Number,Transaction Remarks,Withdrawal Amount(INR),Deposit Amount(INR),Balance(INR)
0,1.0,01-04-2026,01-04-2026,NaN,UPI/HDFC BANK/hdfcbank.billd/Pay/HDFC BANK/609...,8000.00,0.0,116045.48
1,2.0,02-04-2026,02-04-2026,NaN,ACH/TP ACH INDIANESIGN/ICIC7022208230013669/21...,2000.00,0.0,114045.48
2,3.0,02-04-2026,02-04-2026,NaN,UPI/Swiggy Lim/swiggy1online./UPI/AXIS BANK/64...,1392.00,0.0,112653.48
3,4.0,02-04-2026,02-04-2026,NaN,UPI/Google Ind/gpay.bp.rechar/UPI/AXIS BANK/60...,148.90,0.0,112504.58
4,5.0,03-04-2026,03-04-2026,NaN,BIL/Consumer Loan XX91439 EMI Durg,10125.00,0.0,102379.58
5,6.0,04-04-2026,04-04-2026,NaN,UPI/SRI VENKAT/0514700a006723/UPI/Punjab Nat/6...,750.00,0.0,101629.58
6,7.0,04-04-2026,04-04-2026,NaN,UPI/Swiggy Ltd/swiggyupi@axb/UPI/AXIS BANK/646...,1370.00,0.0,100259.58
7,8.0,04-04-2026,04-04-2026,NaN,BIL/NEFT/IN12609431496564/TVH MAHANY/CITY UNIO...,15000.00,0.0,85259.58
8,9.0,04-04-2026,04-04-2026,NaN,BIL/NEFT/IN12609431496812/Rent/V Malini A/UCO ...,8000.00,0.0,77259.58
9,10.0,04-04-2026,04-04-2026,NaN,UPI/srividhyak/srividhyakamal/UPI/UNION BANK/6...,2209.00,0.0,75050.58


In [21]:
df.columns

Index(['ï»¿S No.', 'Value Date', 'Transaction Date', 'Cheque Number',
       'Transaction Remarks', 'Withdrawal Amount(INR)', 'Deposit Amount(INR)',
       'Balance(INR)'],
      dtype='str')

In [22]:
df.drop(['Cheque Number'], axis=1, inplace=True)

In [23]:
df

,ï»¿S No.,Value Date,Transaction Date,Transaction Remarks,Withdrawal Amount(INR),Deposit Amount(INR),Balance(INR)
0,1.0,01-04-2026,01-04-2026,UPI/HDFC BANK/hdfcbank.billd/Pay/HDFC BANK/609...,8000.00,0.0,116045.48
1,2.0,02-04-2026,02-04-2026,ACH/TP ACH INDIANESIGN/ICIC7022208230013669/21...,2000.00,0.0,114045.48
2,3.0,02-04-2026,02-04-2026,UPI/Swiggy Lim/swiggy1online./UPI/AXIS BANK/64...,1392.00,0.0,112653.48
3,4.0,02-04-2026,02-04-2026,UPI/Google Ind/gpay.bp.rechar/UPI/AXIS BANK/60...,148.90,0.0,112504.58
4,5.0,03-04-2026,03-04-2026,BIL/Consumer Loan XX91439 EMI Durg,10125.00,0.0,102379.58
5,6.0,04-04-2026,04-04-2026,UPI/SRI VENKAT/0514700a006723/UPI/Punjab Nat/6...,750.00,0.0,101629.58
6,7.0,04-04-2026,04-04-2026,UPI/Swiggy Ltd/swiggyupi@axb/UPI/AXIS BANK/646...,1370.00,0.0,100259.58
7,8.0,04-04-2026,04-04-2026,BIL/NEFT/IN12609431496564/TVH MAHANY/CITY UNIO...,15000.00,0.0,85259.58
8,9.0,04-04-2026,04-04-2026,BIL/NEFT/IN12609431496812/Rent/V Malini A/UCO ...,8000.00,0.0,77259.58
9,10.0,04-04-2026,04-04-2026,UPI/srividhyak/srividhyakamal/UPI/UNION BANK/6...,2209.00,0.0,75050.58


In [24]:
df["Transaction Remarks"] = df["Transaction Remarks"].str.lower().str.strip()

In [25]:
df.head()

,ï»¿S No.,Value Date,Transaction Date,Transaction Remarks,Withdrawal Amount(INR),Deposit Amount(INR),Balance(INR)
0,1.0,01-04-2026,01-04-2026,upi/hdfc bank/hdfcbank.billd/pay/hdfc bank/609...,8000.0,0.0,116045.48
1,2.0,02-04-2026,02-04-2026,ach/tp ach indianesign/icic7022208230013669/21...,2000.0,0.0,114045.48
2,3.0,02-04-2026,02-04-2026,upi/swiggy lim/swiggy1online./upi/axis bank/64...,1392.0,0.0,112653.48
3,4.0,02-04-2026,02-04-2026,upi/google ind/gpay.bp.rechar/upi/axis bank/60...,148.9,0.0,112504.58
4,5.0,03-04-2026,03-04-2026,bil/consumer loan xx91439 emi durg,10125.0,0.0,102379.58


In [26]:
def categorize(text):
    text = str(text).lower().strip()

    if any(word in text for word in ["swiggy", "zomato", "restaurant", "bigbasket"]):
        return "Food"
    elif any(word in text for word in ["amazon", "flipkart", "meesho", "ajio", "myntra"]):
        return "Shopping"
    elif any(word in text for word in ["uber", "ola", "fuel", "petrol"]):
        return "Transport"
    elif any(word in text for word in ["netflix", "prime", "apple"]):
        return "Entertainment"
    elif any(word in text for word in ["atm", "withdrawal", "cash"]):
        return "Cash Withdrawal"
    elif any(word in text for word in ["salary", "interest"]):
        return "Income"
    elif any(word in text for word in ["indianesign", "eba", "sbilife"]):
        return "Investment"
    elif any(word in text for word in ["emi", "loan","school","rent","apollo"]):
        return "Bills & Payments"
    else:
        return "Others"


In [27]:

# Apply to your dataframe
df["Category"] = df["Transaction Remarks"].apply(categorize)

In [28]:
df

,ï»¿S No.,Value Date,Transaction Date,Transaction Remarks,Withdrawal Amount(INR),Deposit Amount(INR),Balance(INR),Category
0,1.0,01-04-2026,01-04-2026,upi/hdfc bank/hdfcbank.billd/pay/hdfc bank/609...,8000.00,0.0,116045.48,Others
1,2.0,02-04-2026,02-04-2026,ach/tp ach indianesign/icic7022208230013669/21...,2000.00,0.0,114045.48,Investment
2,3.0,02-04-2026,02-04-2026,upi/swiggy lim/swiggy1online./upi/axis bank/64...,1392.00,0.0,112653.48,Food
3,4.0,02-04-2026,02-04-2026,upi/google ind/gpay.bp.rechar/upi/axis bank/60...,148.90,0.0,112504.58,Others
4,5.0,03-04-2026,03-04-2026,bil/consumer loan xx91439 emi durg,10125.00,0.0,102379.58,Bills & Payments
5,6.0,04-04-2026,04-04-2026,upi/sri venkat/0514700a006723/upi/punjab nat/6...,750.00,0.0,101629.58,Others
6,7.0,04-04-2026,04-04-2026,upi/swiggy ltd/swiggyupi@axb/upi/axis bank/646...,1370.00,0.0,100259.58,Food
7,8.0,04-04-2026,04-04-2026,bil/neft/in12609431496564/tvh mahany/city unio...,15000.00,0.0,85259.58,Others
8,9.0,04-04-2026,04-04-2026,bil/neft/in12609431496812/rent/v malini a/uco ...,8000.00,0.0,77259.58,Bills & Payments
9,10.0,04-04-2026,04-04-2026,upi/srividhyak/srividhyakamal/upi/union bank/6...,2209.00,0.0,75050.58,Others


In [29]:
df = df.dropna()

In [30]:
df.columns

Index(['ï»¿S No.', 'Value Date', 'Transaction Date', 'Transaction Remarks',
       'Withdrawal Amount(INR)', 'Deposit Amount(INR)', 'Balance(INR)',
       'Category'],
      dtype='str')

In [31]:
# Features & target
X = df["Transaction Remarks"]
y = df["Category"]

In [32]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
# Pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000))
])

In [34]:
# Train
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [35]:
# Predict
y_pred = model.predict(X_test)

In [36]:
# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7
                  precision    recall  f1-score   support

Bills & Payments       0.00      0.00      0.00         1
            Food       1.00      1.00      1.00         2
      Investment       1.00      1.00      1.00         3
          Others       0.40      1.00      0.57         2
        Shopping       0.00      0.00      0.00         2

        accuracy                           0.70        10
       macro avg       0.48      0.60      0.51        10
    weighted avg       0.58      0.70      0.61        10



c:\Users\Durga\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Durga\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Durga\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [37]:
# Save model
joblib.dump(model, "C:/Users/Durga/OneDrive/Desktop/DataScience/ExpenseAnalyser/expense_model2.pkl")

['C:/Users/Durga/OneDrive/Desktop/DataScience/ExpenseAnalyser/expense_model2.pkl']